# Search for optimal LSTM hyperparameters

In [ ]:
!pip install keras-tuner

import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras import backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd
from sklearn.utils import resample
from sklearn.metrics import recall_score, classification_report

# Reload data if needed (assuming sensor_data and other necessary variables are available from previous cells)
# df = sensor_data.copy()

# Define the model-building function for KerasTuner
def build_lstm_model(hp):
    model = Sequential()
    model.add(Input(shape=(seq_len, len(feature_cols))))

    # Tune the number of LSTM units
    hp_lstm_units = hp.Int('lstm_units', min_value=32, max_value=128, step=32)
    model.add(LSTM(units=hp_lstm_units, return_sequences=False))
    model.add(Dropout(hp.Float('dropout_1', min_value=0.0, max_value=0.5, step=0.1)))

    # Tune the number of dense layers and units
    for i in range(hp.Int('num_dense_layers', min_value=1, max_value=2, step=1)):
        model.add(Dense(units=hp.Int(f'dense_units_{i}', min_value=32, max_value=128, step=32), activation='relu'))
        model.add(Dropout(hp.Float(f'dropout_{i+2}', min_value=0.0, max_value=0.5, step=0.1)))

    model.add(Dense(1, activation='sigmoid'))

    # Tune the learning rate for the optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
             tf.keras.metrics.Recall(name='recall') # Use Keras Recall metric for tuning
        ]
    )

    return model

# Define the tuner
# Use Hyperband as it's efficient for finding good hyperparameters quickly
tuner = kt.Hyperband(
    build_lstm_model,
    objective=kt.Objective('recall', direction='max'), # Maximize recall
    max_epochs=20, # Maximum number of epochs to train a model
    factor=3, # Factor by which the number of epochs is increased
    hyperband_iterations=2, # Number of times to iterate Hyperband
    directory='keras_tuner_dir', # Directory to save results
    project_name='lstm_hyperparameter_tuning'
)

# Create patient-level labels for stratification (assuming this is already done in a previous cell)
# grouped = df.groupby('patient_id')
# patient_labels = {pid: int(grouped.get_group(pid)[label_col].any()) for pid in df['patient_id'].unique()}
# pids = np.array(list(patient_labels.keys()))
# labels = np.array(list(patient_labels.values()))

# Initialize StratifiedGroupKFold (assuming this is already done in a previous cell)
# gkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=32)

# --- Prepare data for Tuning (using the first fold for demonstration) ---
# In a real scenario, you might want to use a separate tuning set or inner cross-validation.
# For simplicity here, we'll use the first split from the cross-validation defined earlier.
# This is just for the tuning search; the final evaluation will be on the test sets of the outer CV.

# Find the first split (assuming pids and labels are defined)
train_patient_indices, val_patient_indices = list(gkf.split(pids, labels, groups=pids))[0]
train_pids_tune, val_pids_tune = pids[train_patient_indices], pids[val_patient_indices]

df_train_tune = df[df['patient_id'].isin(train_pids_tune)].copy()
df_val_tune = df[df['patient_id'].isin(val_pids_tune)].copy()

# Oversample Minority Class in Training for tuning
df_train_pos_tune = df_train_tune[df_train_tune[label_col] == True]
df_train_neg_tune = df_train_tune[df_train_tune[label_col] == False]

if len(df_train_pos_tune) > 0 and len(df_train_neg_tune) > 0:
    df_train_pos_oversampled_tune = resample(
        df_train_pos_tune,
        replace=True,
        n_samples=len(df_train_neg_tune),
        random_state=42
    )
    df_train_balanced_tune = pd.concat([df_train_neg_tune, df_train_pos_oversampled_tune]).sort_index()
else:
    print("Warning: Tuning training set missing one class, skipping oversampling.")
    df_train_balanced_tune = df_train_tune.copy()

# Preprocessing & Scaling (FIT ON TRAIN_TUNE, TRANSFORM ON TRAIN_TUNE & VAL_TUNE)
scaler_tune = StandardScaler()
df_train_balanced_tune[feature_cols] = scaler_tune.fit_transform(df_train_balanced_tune[feature_cols])
df_val_tune[feature_cols] = scaler_tune.transform(df_val_tune[feature_cols])

# Generate Sequences for tuning
X_train_tune, y_train_tune = create_sequences(df_train_balanced_tune, feature_cols, label_col, seq_len)
X_val_tune, y_val_tune = create_sequences(df_val_tune, feature_cols, label_col, seq_len)


# Start the search
print("Starting hyperparameter tuning search...")
tuner.search(
    X_train_tune, y_train_tune,
    validation_data=(X_val_tune, y_val_tune),
    epochs=20, # Note: max_epochs is set in the tuner, this epoch value here
               # is the number of epochs to train *each* candidate model for
               # within the Hyperband process. Should be <= max_epochs in tuner.
    batch_size=64
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"\nOptimal Hyperparameters:")
print(f"LSTM Units: {best_hps.get('lstm_units')}")
print(f"Dropout 1: {best_hps.get('dropout_1'):.4f}")
print(f"Number of Dense Layers: {best_hps.get('num_dense_layers')}")
for i in range(best_hps.get('num_dense_layers')):
    print(f"Dense Units {i}: {best_hps.get(f'dense_units_{i}')}")
    print(f"Dropout {i+2}: {best_hps.get(f'dropout_{i+2}'):.4f}")
print(f"Learning Rate: {best_hps.get('learning_rate'):.4f}")

# You can now use best_hps to build the final model for evaluation on the test sets
# (This part is not included in this cell but would be the next step in the plan)